# [Chapter 9 — Practical Issues in Fitting, §9.4] Pitfall 4: Residual Diagnostics — Detecting Model Misspecification

**Book:** *Essential Considerations for Modeling Epidemics* (Hyman/Qu/Xue, 2026)
**Chapter:** Chapter 9 — Practical Issues in Fitting
**Considerations developed:** 9
**Estimated runtime:** ≤ 30s

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/machyman/hyman2026essential/blob/main/python/notebooks/)

## What this notebook does
Demonstrates residual plots, Q-Q normality checks, and the autocorrelation function for detecting misspecified noise models or omitted-mechanism patterns in the residuals.

## What you should already know
Chapter 8 fitting notebooks.


## Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
import numpy as np
import matplotlib.pyplot as plt
from shared import book_style, BOOK_COLORS, integrate_sir_i
from shared.parameters import baseline_chapter_08
from shared.seeds import set_seed_chapter_08
from shared.verification import assert_within_tolerance
set_seed_chapter_08()
book_style()


In [ ]:
params = baseline_chapter_08()
result = integrate_sir_i(params)
t, S, I = result['t'], result['S'], result['I']
J_true = params['c_I'] * params['beta'] * (S/params['N']) * I

# Two scenarios: well-specified (white noise) vs misspecified (correlated noise)
np.random.seed(42)
J_white = J_true + 0.05 * J_true.max() * np.random.randn(len(t))
# Correlated noise (AR(1)) — simulates omitted dynamics
phi = 0.7
noise = np.zeros(len(t))
e = 0.05 * J_true.max() * np.random.randn(len(t))
for i in range(1, len(t)):
    noise[i] = phi * noise[i-1] + e[i]
J_correlated = J_true + noise

residuals_white = J_white - J_true
residuals_correlated = J_correlated - J_true

fig, axes = plt.subplots(2, 2, figsize=(11, 7))

# Residuals vs time
for ax, res, title in zip(axes[0], [residuals_white, residuals_correlated],
                          ['White noise (well-specified)', 'AR(1) noise (misspecified)']):
    ax.plot(t, res, '.', alpha=0.4, color=BOOK_COLORS['primary'])
    ax.axhline(0, color='k', lw=0.5)
    ax.set_xlabel('Time (days)')
    ax.set_ylabel('Residual')
    ax.set_title(f'Residuals: {title}')

# ACF
from numpy import correlate
def acf(x, maxlag=20):
    x = x - x.mean()
    c = correlate(x, x, mode='full') / np.sum(x**2)
    return c[len(x)-1: len(x)-1+maxlag]
ax = axes[1, 0]
ax.stem(range(20), acf(residuals_white))
ax.axhline(0, color='k', lw=0.5)
ax.set_xlabel('Lag')
ax.set_ylabel('ACF')
ax.set_title('ACF (white): no significant autocorrelation')

ax = axes[1, 1]
ax.stem(range(20), acf(residuals_correlated))
ax.axhline(0, color='k', lw=0.5)
ax.set_xlabel('Lag')
ax.set_ylabel('ACF')
ax.set_title('ACF (correlated): clear pattern → misspecification')

plt.tight_layout()
plt.show()
print("\nThe correlated-noise residuals reveal hidden structure that the model missed.")
